In [138]:
pip install sportsdataverse polars numpy pandas matplotlib scipy pymc

Note: you may need to restart the kernel to use updated packages.


In [139]:
import sportsdataverse.nhl as nhl
import polars as pl
import pandas as pd
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import expit

import matplotlib.pyplot as plt

In [161]:
pbp = nhl.load_nhl_pbp(list(range(2022, 2027))).filter(
          pl.col("event_type").is_in(["SHOT", "MISSED_SHOT", "GOAL"])
     ).with_columns(
          pl.col("event_type")
               .replace_strict({"SHOT":0, "MISSED_SHOT":0, "GOAL":1}, return_dtype=pl.UInt8)
               .alias("outcome")
     ).with_columns(
         (pl.col("outcome") - pl.col("xg")).alias("GAx")
     ).select(["event_player_1_id", "event_player_1_name", "xg", "outcome", "GAx"])
pbp

event_player_1_id,event_player_1_name,xg,outcome,GAx
i64,str,f64,u8,f64
8474564,"""Steven Stamkos""",0.0219,0,-0.0219
8476934,"""Brock McGinn""",0.0838,0,-0.0838
8480172,"""Jan Rutta""",0.0041,0,-0.0041
8478046,"""Danton Heinen""",0.1237,0,-0.1237
8476292,"""Ondrej Palat""",0.1193,0,-0.1193
…,…,…,…,…
8477447,"""Shea Theodore""",0.0576,0,-0.0576
8475913,"""Mark Stone""",0.2396,0,-0.2396
8477940,"""Nikolaj Ehlers""",0.797,1,0.203


In [162]:
shots = pbp.filter(
    pl.col("xg").is_not_null() & pl.col("event_player_1_id").is_not_null()
)

In [166]:
mu0 = shots["GAx"].mean()                               # league average GAx per shot
sigma2 = (shots["xg"] * (1 - shots["xg"])).mean()       # luck per shot

players = shots.group_by(["event_player_1_id", "event_player_1_name"]).agg(
    n=pl.len().cast(pl.Float64), #number of shot attempts
    gax_per_shot=pl.col("GAx").mean(),
)

est = players.filter(pl.col("n") >= 100)
sigma0_sq = est["gax_per_shot"].var() - (sigma2 / est["n"]).mean()

c = sigma2 / sigma0_sq
print(f"mu0={mu0:.4f}  sigma2={sigma2:.4f}  sigma0_sq={sigma0_sq:.6f}  c={c:.0f}")

players = players.with_columns(
    gax_regressed=(pl.col("n") * pl.col("gax_per_shot") + c * mu0) / (pl.col("n") + c)    
).sort("gax_regressed", descending=True)

players = players.with_columns(
    goals_above_avg_per_100_SAT=(pl.col("gax_regressed") - mu0)*100, # goals above average per 100 shot attempts
    rank= pl.col("gax_regressed").rank(method="ordinal", descending=True)
)

with pl.Config(tbl_rows=-1):
    print(players.head(15))

mu0=-0.0170  sigma2=0.0711  sigma0_sq=0.000071  c=1007
shape: (15, 7)
┌─────────────────┬────────────────┬────────┬──────────────┬───────────────┬────────────────┬──────┐
│ event_player_1_ ┆ event_player_1 ┆ n      ┆ gax_per_shot ┆ gax_regressed ┆ goals_above_av ┆ rank │
│ id              ┆ _name          ┆ ---    ┆ ---          ┆ ---           ┆ g_per_100_SAT  ┆ ---  │
│ ---             ┆ ---            ┆ f64    ┆ f64          ┆ f64           ┆ ---            ┆ u32  │
│ i64             ┆ str            ┆        ┆              ┆               ┆ f64            ┆      │
╞═════════════════╪════════════════╪════════╪══════════════╪═══════════════╪════════════════╪══════╡
│ 8477934         ┆ Leon Draisaitl ┆ 1996.0 ┆ 0.012776     ┆ 0.002781      ┆ 1.981318       ┆ 1    │
│ 8479420         ┆ Tage Thompson  ┆ 1897.0 ┆ 0.009402     ┆ 0.000236      ┆ 1.726831       ┆ 2    │
│ 8480027         ┆ Jason          ┆ 1949.0 ┆ 0.007544     ┆ -0.000828     ┆ 1.620429       ┆ 3    │
│                 ┆ R